In [1]:
import pandas as pd
import numpy as np


### 1. Loading Datasets

In [5]:
users=pd.read_csv('C:/CLV-Project/data/raw/users.csv')
bookings=pd.read_csv('C:/CLV-Project/data/raw/bookings (1).csv')
chats=pd.read_csv('C:/CLV-Project/data/raw/chatbot_logs.csv')
reviews=pd.read_csv('C:/CLV-Project/data/raw/reviews.csv')
snacks=pd.read_csv('C:/CLV-Project/data/raw/concession_orders (1).csv')

print("All 5 datasets loaded successfully.")
print(f"   Users: {users.shape[0]} rows, {users.shape[1]} cols")
print(f"   Bookings: {bookings.shape[0]} rows, {bookings.shape[1]} cols")
print(f"   Snack Orders: {snacks.shape[0]} rows, {snacks.shape[1]} cols")
print(f"   Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} cols")
print(f"   Chatbot Logs: {chats.shape[0]} rows, {chats.shape[1]} cols")


All 5 datasets loaded successfully.
   Users: 46 rows, 4 cols
   Bookings: 215 rows, 8 cols
   Snack Orders: 150 rows, 6 cols
   Reviews: 42 rows, 6 cols
   Chatbot Logs: 65 rows, 5 cols


### 2. Initial Data Inspection

In [6]:
datasets = {
    'Users': users,
    'Bookings': bookings,
    'Snack Orders': snacks,
    'Reviews': reviews,
    'Chatbot Logs': chats
}
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"📋 {name.upper()} — Shape: {df.shape}")
    print(f"{'='*50}")
    print("\nColumn Types:")
    print(df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())


📋 USERS — Shape: (46, 4)

Column Types:
id           int64
username    object
email       object
is_admin     int64

First 3 rows:
   id      username                     email  is_admin
0   1         admin      admin@cinesphere.com         1
1   2  sarah_connor      sarah.connor@sky.net         0
2   3    tony_stark  tony@starkindustries.com         0

📋 BOOKINGS — Shape: (215, 8)

Column Types:
id              int64
user_id         int64
theater        object
showtime       object
tickets         int64
total         float64
created_at     object
username       object

First 3 rows:
   id  user_id         theater  showtime  tickets  total           created_at      username
0   1        2     IMAX Nebula  10:45 PM        2   29.0  2026-01-22 20:15:00  sarah_connor
1   2        2  Apollo Theater  10:45 PM        2   29.0  2026-02-02 15:45:00  sarah_connor
2   3        2     IMAX Nebula   8:00 PM        4   58.0  2026-03-04 16:20:00  sarah_connor

📋 SNACK ORDERS — Shape: (150, 6)

Colum

### 3. Data Quality Audit

In [ ]:
print("\n" + "=" * 60)
print("🔍 DATA QUALITY AUDIT REPORT")
print("=" * 60)
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    
    # Null analysis
    null_counts = df.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    if len(null_cols) > 0:
        print("  ⚠️  Null values found:")
        for col, count in null_cols.items():
            pct = (count / len(df)) * 100
            print(f"      {col}: {count} nulls ({pct:.1f}%)")
    else:
        print("  ✅ No null values.")

    # Duplicate analysis
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        print(f"  ⚠️  {dup_count} duplicate rows detected.")
    else:
        print("  ✅ No duplicate rows.")

    # Numeric range check
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("  📐 Numeric ranges:")
        for col in numeric_cols:
            print(f"      {col}: min={df[col].min()}, max={df[col].max()}, mean={df[col].mean():.2f}")


🔍 DATA QUALITY AUDIT REPORT

--- USERS ---
  ✅ No null values.
  ✅ No duplicate rows.
  📐 Numeric ranges:
      id: min=1, max=46, mean=23.50
      is_admin: min=0, max=1, mean=0.02

--- BOOKINGS ---
  ✅ No null values.
  ✅ No duplicate rows.
  📐 Numeric ranges:
      id: min=1, max=215, mean=108.00
      user_id: min=2, max=46, mean=17.27
      tickets: min=1, max=4, mean=2.30
      total: min=14.5, max=58.0, mean=33.32

--- SNACK ORDERS ---
  ✅ No null values.
  ✅ No duplicate rows.
  📐 Numeric ranges:
      id: min=1, max=150, mean=75.50
      user_id: min=2, max=44, mean=14.74
      total: min=4.0, max=26.0, mean=13.48

--- REVIEWS ---
  ✅ No null values.
  ✅ No duplicate rows.
  📐 Numeric ranges:
      id: min=1, max=42, mean=21.50
      user_id: min=2, max=40, mean=15.45
      rating: min=2, max=5, mean=4.24

--- CHATBOT LOGS ---
  ✅ No null values.
  ✅ No duplicate rows.
  📐 Numeric ranges:
      id: min=1, max=65, mean=33.00
      user_id: min=2, max=46, mean=18.62


### 4. Data Cleaning and Standardisation

In [14]:
#User Table
users_clean = users[users['is_admin'] == 0].copy()
users_clean['id'] = users_clean['id'].astype(int)

print(f"  ✅ Users: Removed admin account. {len(users_clean)} customer records retained.")

  ✅ Users: Removed admin account. 45 customer records retained.


In [15]:
#Booking Table
bookings_clean=bookings.copy()
bookings_clean['created_at']=pd.to_datetime(bookings_clean['created_at'])
bookings_clean['user_id']=bookings_clean['user_id'].astype(int)
bookings_clean['tickets']=bookings_clean['tickets'].astype(int)
bookings_clean['total']=bookings_clean['total'].astype(float)

## Derieved Columns
bookings_clean['booking_month']=bookings_clean['created_at'].dt.to_period('M').astype(str)
bookings_clean['booking_day']=bookings_clean['created_at'].dt.day_name()
bookings_clean['booking_hour']=bookings_clean['created_at'].dt.hour
print(f"  ✅ Bookings: Parsed dates. Added month, day, hour columns. {len(bookings_clean)} records.")

  ✅ Bookings: Parsed dates. Added month, day, hour columns. 215 records.


In [18]:
import json

In [20]:
# Snacks Table
snacks_clean=snacks.copy()
snacks_clean['created_at']=pd.to_datetime(snacks_clean['created_at'])
snacks_clean['user_id']=snacks_clean['user_id'].astype(int)
snacks_clean['total']=snacks_clean['total'].astype(float)
# Parse the JSON items column into a proper Python list
def safe_parse_items(x):
    """Safely parse items field which can be a JSON string or already a list."""
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except json.JSONDecodeError:
            try:
                import ast
                return ast.literal_eval(x)
            except:
                return [x]
    return [str(x)]
snacks_clean['items_list']  = snacks_clean['items'].apply(safe_parse_items)
snacks_clean['item_count']  = snacks_clean['items_list'].apply(len)
snacks_clean['order_month'] = snacks_clean['created_at'].dt.to_period('M').astype(str)
print(f"  ✅ Snacks: Parsed JSON items. Added item_count column. {len(snacks_clean)} records.")

  ✅ Snacks: Parsed JSON items. Added item_count column. 150 records.


In [21]:
# Reviews Table
reviews_clean = reviews.copy()
reviews_clean['created_at'] = pd.to_datetime(reviews_clean['created_at'])
reviews_clean['user_id']    = reviews_clean['user_id'].astype(int)
reviews_clean['rating']     = reviews_clean['rating'].astype(int)
reviews_clean['comment']    = reviews_clean['comment'].fillna('')
# Derived: comment length (useful for NLP features later)
reviews_clean['comment_length'] = reviews_clean['comment'].apply(len)
print(f"  ✅ Reviews: Parsed dates. Added comment_length. {len(reviews_clean)} records.")


  ✅ Reviews: Parsed dates. Added comment_length. 42 records.


In [22]:
chats_clean = chats.copy()
chats_clean['created_at'] = pd.to_datetime(chats_clean['created_at'])
# Anonymous/guest user IDs (null) → fill with 0
chats_clean['user_id'] = chats_clean['user_id'].fillna(0).astype(int)
# Parse bot_response JSON if it's a string
def safe_parse_response(x):
    """Parse bot response JSON to extract the response type."""
    if isinstance(x, dict):
        return x.get('type', 'unknown')
    if isinstance(x, str):
        try:
            parsed = json.loads(x)
            return parsed.get('type', 'unknown')
        except:
            return 'text'
    return 'unknown'
chats_clean['response_type'] = chats_clean['bot_response'].apply(safe_parse_response)
chats_clean['query_length']  = chats_clean['user_message'].apply(len)
print(f"  ✅ Chats: Parsed response types. Added query_length. {len(chats_clean)} records.")


  ✅ Chats: Parsed response types. Added query_length. 65 records.


### 6. Saving Cleaned Datasets

In [26]:
import os
cleaned_dir='C:/CLV-Project/data/cleaned_data_'
users_clean.to_csv(os.path.join(cleaned_dir,'users_clean.csv'),index=False)
bookings_clean.to_csv(os.path.join(cleaned_dir,'bookings_clean.csv'),index=False)
snacks_clean.to_csv(os.path.join(cleaned_dir,'snacks_clean.csv'),index=False)
reviews_clean.to_csv(os.path.join(cleaned_dir,'reviews_clean.csv'),index=False)
chats_clean.to_csv(os.path.join(cleaned_dir,'chats_clean.csv'),index=False)

### Final Summary

In [27]:

total_booking_rev = bookings_clean['total'].sum()
total_snack_rev   = snacks_clean['total'].sum()
total_revenue     = total_booking_rev + total_snack_rev

In [29]:

print("=" * 60)
print(f"")
print(f"  📁 Raw Files Loaded From  : ../data/raw/")
print(f"  📁 Clean Files Saved To   : ../data/cleaned/")
print(f"")
print(f"  👤 Active Customers       : {len(users_clean)}")
print(f"  🎫 Total Bookings         : {len(bookings_clean)}")
print(f"  🍿 Total Snack Orders     : {len(snacks_clean)}")
print(f"  ⭐ Total Reviews           : {len(reviews_clean)}")
print(f"  💬 Total Chat Sessions    : {len(chats_clean)}")
print(f"")
print(f"  💰 Total Ticket Revenue   : ${total_booking_rev:,.2f}")
print(f"  💰 Total Snack Revenue    : ${total_snack_rev:,.2f}")
print(f"  💰 Combined Revenue       : ${total_revenue:,.2f}")
print(f"  💰 Avg Revenue per User   : ${total_revenue / len(users_clean):,.2f}")
print(f"")
print(f"  ⭐ Average Review Rating   : {reviews_clean['rating'].mean():.2f} / 5.0")
print(f"  📅 Data Range             : {bookings_clean['created_at'].min().strftime('%b %d, %Y')} → {bookings_clean['created_at'].max().strftime('%b %d, %Y')}")
print(f"")
print("=" * 60)

print("=" * 60)


  📁 Raw Files Loaded From  : ../data/raw/
  📁 Clean Files Saved To   : ../data/cleaned/

  👤 Active Customers       : 45
  🎫 Total Bookings         : 215
  🍿 Total Snack Orders     : 150
  ⭐ Total Reviews           : 42
  💬 Total Chat Sessions    : 65

  💰 Total Ticket Revenue   : $7,163.00
  💰 Total Snack Revenue    : $2,022.00
  💰 Combined Revenue       : $9,185.00
  💰 Avg Revenue per User   : $204.11

  ⭐ Average Review Rating   : 4.24 / 5.0
  📅 Data Range             : Jan 05, 2026 → Jul 31, 2026

